In [1]:
import torch

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(num_inputs, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, num_outputs)
        )

    def forward(self, x):
        return self.layers(x)

In [3]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


class ToyDataset(Dataset):
    def __getitem__(self, index):
        return self.features[index], self.labels[index]
    def __init__(self, X, y):
        self.features = X
        self.labels = y
    def __len__(self):
        return len(self.features)    


In [7]:
X_train = torch.tensor(
    [
        [-1.2, 3.1],
        [-0.9, 2.9],
        [-0.5, 2.6],
        [2.3, -1.1],
        [2.7, -1.5]
    ]
)
y_train = torch.tensor([0, 0, 0, 1, 1])


X_test = torch.tensor(
    [
        [-0.8, 2.8],
        [2.6, -1.6]
    ]
)
y_test = torch.tensor([0, 1])

torch.manual_seed(123)

train_dataset = ToyDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, drop_last=True)

test_dataset = ToyDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [14]:
import torch
import torch.nn.functional as F


torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        logits = model(features)

        loss = F.cross_entropy(logits, labels) # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.86
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.01
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.01
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.02
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.01
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.01


In [26]:
model.eval()
with torch.no_grad():
    logits = model(X_train)
    
logits

tensor([[ 3.3279, -2.6567],
        [ 3.0929, -2.4681],
        [ 2.7465, -2.1870],
        [-2.1945,  1.7960],
        [-2.7381,  2.1805]])

In [32]:
probas = torch.nn.functional.softmax(logits, dim=-1)
probas

tensor([[0.9975, 0.0025],
        [0.9962, 0.0038],
        [0.9929, 0.0071],
        [0.0182, 0.9818],
        [0.0073, 0.9927]])

In [33]:
torch.argmax(probas, dim=0)

tensor([0, 4])

In [38]:
probas = torch.argmax(logits, dim=1)

In [39]:
probas == y_train

tensor([True, True, True, True, True])

In [40]:
y_train

tensor([0, 0, 0, 1, 1])

In [41]:
probas

tensor([0, 0, 0, 1, 1])

In [43]:
def compute_accuracy(model, data_loader):
    correct_pred, num_examples = 0, 0
    for features, targets in data_loader:
        logits = model(features)
        probas = torch.nn.functional.softmax(logits, dim=1)
        predicted_labels = torch.argmax(probas, dim=1)
        num_examples += targets.size(0)
        correct_pred += (predicted_labels == targets).sum()
    return correct_pred.float()/num_examples * 100

In [44]:
compute_accuracy(model, train_loader)

tensor(100.)

In [45]:
compute_accuracy(model, test_loader)

tensor(100.)

In [47]:
torch.save(model.state_dict(), 'model.pt')

In [49]:
model.load_state_dict(torch.load('model.pt', weights_only= True))

<All keys matched successfully>